# Bank Personal Loan Prediction: End-to-End Master Workflow

**Comprehensive Machine Learning Pipeline: Data Collection $\rightarrow$ EDA $\rightarrow$ Feature Engineering $\rightarrow$ Preprocessing $\rightarrow$ Cross-Validation $\rightarrow$ Hyperparameter Tuning $\rightarrow$ Evaluation $\rightarrow$ Inference**

---

### Executive Summary
Thera Bank seeks to identify retail depositors with high propensity to convert into personal loan customers.
This notebook runs the complete reproducible machine learning workflow comparing only **Random Forest**, **XGBoost**, and **CatBoost** under 5-Fold Stratified Cross-Validation and calibrated decision thresholding.

In [ ]:
import os
import sys
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure environment and project path
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import (
    RAW_DATA_PATH, MODEL_PATH, MODEL_METADATA_PATH,
    TARGET_COL, RANDOM_SEED, CV_FOLDS
)
from src.data.data_loader import load_raw_data, load_and_split_data, save_processed_data
from src.data.validation import validate_raw_data
from src.features.feature_engineering import LoanFeatureEngineer, clean_experience_column
from src.features.preprocessing import create_full_pipeline
from src.models.train import get_candidate_models, cross_validate_model
from src.models.tune import tune_random_forest, tune_xgboost, tune_catboost
from src.models.evaluation import evaluate_model, find_optimal_threshold, compare_models_metrics
from src.utils.plotting import (
    plot_target_distribution, plot_correlation_matrix,
    plot_roc_pr_curves, plot_confusion_matrix, plot_threshold_analysis
)

sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")

## Step 1: Data Ingestion & Schema Validation

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
is_valid, issues = validate_raw_data(df_raw)
print(f"Data validation passed: {is_valid} (Found {len(issues)} issues)")
print(f"Dataset shape: {df_raw.shape}")
display(df_raw.head(3))

## Step 2: Exploratory Data Analysis & Target Balance

In [ ]:
target_dist = df_raw[TARGET_COL].value_counts(normalize=True) * 100
print(f"Class 0: {target_dist[0]:.2f}% | Class 1: {target_dist[1]:.2f}%")

fig = plot_target_distribution(df_raw, target_col=TARGET_COL, save_name=None)
plt.show()

## Step 3: Stratified Splitting (Zero Data Leakage)

In [ ]:
X_train, X_test, y_train, y_test = load_and_split_data(RAW_DATA_PATH, test_size=0.20, random_seed=RANDOM_SEED)
save_processed_data(X_train, X_test, y_train, y_test)
print(f"Train Shape: {X_train.shape} (Positive rate: {y_train.mean():.2%})")
print(f"Test Shape:  {X_test.shape} (Positive rate: {y_test.mean():.2%})")

## Step 4: Feature Engineering & Preprocessing Verification

In [ ]:
fe = LoanFeatureEngineer(drop_identifiers=True)
sample_trans = fe.fit_transform(X_train.head(5))
print("Engineered feature columns added:")
print([col for col in sample_trans.columns if col not in X_train.columns])
display(sample_trans.head(3))

## Step 5: 5-Fold Stratified Cross-Validation (RF, XGBoost, CatBoost)

In [ ]:
candidates = get_candidate_models(random_state=RANDOM_SEED)
cv_results = {}

for name, model in candidates.items():
    print(f"Cross-validating {name}...")
    summary = cross_validate_model(name, model, X_train, y_train, n_splits=CV_FOLDS)
    cv_results[name] = summary

cv_table = pd.DataFrame([{
    "Model": name,
    "CV F1 (Mean)": s["cv_f1_mean"],
    "CV PR-AUC (Mean)": s["cv_pr_auc_mean"],
    "CV ROC-AUC (Mean)": s["cv_roc_auc_mean"],
    "CV Accuracy (Mean)": s["cv_accuracy_mean"],
} for name, s in cv_results.items()]).sort_values("CV PR-AUC (Mean)", ascending=False)

display(cv_table)

## Step 6: Hyperparameter Tuning

In [ ]:
print("Tuning Random Forest...")
rf_pipe, rf_params, _ = tune_random_forest(X_train, y_train, n_iter=6)

print("Tuning XGBoost...")
xgb_pipe, xgb_params, _ = tune_xgboost(X_train, y_train, n_iter=6)

print("Tuning CatBoost...")
cat_pipe, cat_params, _ = tune_catboost(X_train, y_train, n_iter=5)

tuned_pipelines = {
    "RandomForest": rf_pipe,
    "XGBoost": xgb_pipe,
    "CatBoost": cat_pipe,
}

## Step 7: Final Evaluation on Reserved Test Set

In [ ]:
test_metrics = {}
pred_dict = {}

for name, pipe in tuned_pipelines.items():
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    test_metrics[name] = evaluate_model(y_test, preds, probs, model_name=name)
    pred_dict[name] = (preds, probs)

comp_df = compare_models_metrics(test_metrics)
print("FINAL TEST SET PERFORMANCE COMPARISON:")
display(comp_df)

## Step 8: Comparative Visualizations

In [ ]:
fig = plot_roc_pr_curves(pred_dict, y_test.values, save_name=None)
plt.show()

## Step 9: Decision Threshold Optimization

In [ ]:
best_model_name = comp_df.iloc[0]["Model"]
best_pipe = tuned_pipelines[best_model_name]
_, best_probs = pred_dict[best_model_name]

fig, opt_thresh = plot_threshold_analysis(y_test.values, best_probs, save_name=None)
plt.show()

calibrated_preds = (best_probs >= opt_thresh).astype(int)
calibrated_eval = evaluate_model(y_test, calibrated_preds, best_probs, model_name=f"{best_model_name} (Calibrated)")
print(f"Optimal Threshold: {opt_thresh:.3f}")
print("Calibrated Metrics:", calibrated_eval)

## Step 10: Model Persistence & Verification

In [ ]:
joblib.dump(best_pipe, MODEL_PATH)
print("Best pipeline persisted to:", MODEL_PATH)

# Test reload and predict
reloaded = joblib.load(MODEL_PATH)
sample_test = X_test.iloc[:5]
sample_pred = reloaded.predict(sample_test)
sample_prob = reloaded.predict_proba(sample_test)[:, 1]

res_df = sample_test.copy()
res_df["Predicted_Class"] = sample_pred
res_df["Probability"] = np.round(sample_prob, 4)
display(res_df[["Age", "Income", "CCAvg", "Education", "Predicted_Class", "Probability"]])